In [4]:
import os
import pandas as pd

raw_dir = "../data/raw"
files = sorted([f for f in os.listdir(raw_dir) if f.endswith(".csv")])
print("CSV files found:", files)

for f in files:
    print("\n" + "=" * 60)
    print(f"FILE: {f}")
    print("=" * 60)
    
    df = pd.read_csv(os.path.join(raw_dir, f), encoding="utf-8")
    print("Shape (rows, columns):", df.shape)
    print("Columns:", df.columns.tolist())
    
    print("\nMissing values:")
    print(df.isnull().sum())
    
    # Print value counts for any categorical/language-like columns
    for col in df.columns:
        if df[col].nunique() <= 20:
            print(f"\nUnique values in '{col}':")
            print(df[col].value_counts(dropna=False))
            
    print("\nFirst 3 rows:")
    display(df.head(3))

CSV files found: ['grievances_holdout_templates.csv', 'grievances_synthetic.csv']

FILE: grievances_holdout_templates.csv
Shape (rows, columns): (614, 3)
Columns: ['grievance_id', 'text', 'category']

Missing values:
grievance_id    0
text            0
category        0
dtype: int64

Unique values in 'category':
category
Corruption & Bribery                   50
Land Records & Revenue                 50
Employment & Labour                    50
Education & Schools                    48
Pension & Provident Fund               47
Banking & Financial Services           46
Police & Law and Order                 46
Healthcare & Hospitals                 45
Water Supply                           43
Ration & Public Distribution System    41
Sanitation & Garbage                   39
Electricity                            37
Municipal Certificates                 36
Roads & Infrastructure                 36
Name: count, dtype: int64

First 3 rows:


,grievance_id,text,category
0,GRVH900075,"Sir/Madam, Patna mein bijli two weeks se gayab...",Electricity
1,GRVH900500,"Namaste sir, Ration dealer Adhartal mein har m...",Corruption & Bribery
2,GRVH900036,"Drinking water contaminated hai Sector 7 mein,...",Water Supply



FILE: grievances_synthetic.csv
Shape (rows, columns): (3110, 3)
Columns: ['grievance_id', 'text', 'category']

Missing values:
grievance_id    0
text            0
category        0
dtype: int64

Unique values in 'category':
category
Employment & Labour                    254
Corruption & Bribery                   252
Land Records & Revenue                 252
Education & Schools                    244
Pension & Provident Fund               235
Police & Law and Order                 234
Banking & Financial Services           233
Healthcare & Hospitals                 229
Water Supply                           217
Ration & Public Distribution System    207
Sanitation & Garbage                   199
Electricity                            186
Roads & Infrastructure                 184
Municipal Certificates                 184
Name: count, dtype: int64

First 3 rows:


,grievance_id,text,category
0,GRV101340,"Eve teasing Adhartal area mein badh gayi hai, ...",Police & Law and Order
1,GRV102878,श्रमिक कार्ड Ranchi में a month से renewal ke ...,Employment & Labour
2,GRV100497,"Dear officer, सड़क पर बिना बैरिकेड के गड्ढा खो...",Roads & Infrastructure


In [5]:
import pandas as pd

df_syn = pd.read_csv("../data/raw/grievances_synthetic.csv", encoding="utf-8")
df_hold = pd.read_csv("../data/raw/grievances_holdout_templates.csv", encoding="utf-8")

print("=== ALL CATEGORIES IN grievances_synthetic.csv ===")
for cat, count in df_syn["category"].value_counts().items():
    print(f"{cat:<40} : {count}")

print("\n=== ALL CATEGORIES IN grievances_holdout_templates.csv ===")
for cat, count in df_hold["category"].value_counts().items():
    print(f"{cat:<40} : {count}")

=== ALL CATEGORIES IN grievances_synthetic.csv ===
Employment & Labour                      : 254
Corruption & Bribery                     : 252
Land Records & Revenue                   : 252
Education & Schools                      : 244
Pension & Provident Fund                 : 235
Police & Law and Order                   : 234
Banking & Financial Services             : 233
Healthcare & Hospitals                   : 229
Water Supply                             : 217
Ration & Public Distribution System      : 207
Sanitation & Garbage                     : 199
Electricity                              : 186
Roads & Infrastructure                   : 184
Municipal Certificates                   : 184

=== ALL CATEGORIES IN grievances_holdout_templates.csv ===
Corruption & Bribery                     : 50
Land Records & Revenue                   : 50
Employment & Labour                      : 50
Education & Schools                      : 48
Pension & Provident Fund                 : 47
B

In [6]:
import re
import pandas as pd

# Combine both Kaggle files
df_all = pd.concat([
    pd.read_csv("../data/raw/grievances_synthetic.csv", encoding="utf-8"),
    pd.read_csv("../data/raw/grievances_holdout_templates.csv", encoding="utf-8")
], ignore_index=True)

# 1. Simple script/language detector
def detect_script(text):
    text = str(text)
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[A-Za-z]", text))
    if has_devanagari and not has_latin:
        return "Hindi (Pure Devanagari)"
    elif has_devanagari and has_latin:
        return "Mixed (Devanagari + Roman)"
    else:
        return "Roman (English or Hinglish)"

df_all["script_type"] = df_all["text"].apply(detect_script)

# 2. Check script distribution for the 4 direct matches
direct_cats = ["Sanitation & Garbage", "Roads & Infrastructure", "Water Supply", "Municipal Certificates"]
print("=== SCRIPT BREAKDOWN FOR 4 DIRECT MATCHES ===")
print(pd.crosstab(df_all[df_all["category"].isin(direct_cats)]["category"], df_all["script_type"]))

# 3. Check for hidden Streetlights, Drainage, and Public Health rows
keywords = {
    "Streetlights": r"street\s*light|streetlight|lamp|pole|स्ट्रीट\s*लाइट|खंभा",
    "Drainage & Sewage": r"drain|sewer|sewage|naali|nali|waterlog|नाली|सीवर|जलभराव",
    "Public Health & Hygiene": r"mosquito|fogging|stray|dengue|malaria|hygiene|मच्छर|फॉगिंग|डेंगू"
}

print("\n=== HIDDEN KEYWORD MATCHES ACROSS ENTIRE DATASET ===")
for target, pattern in keywords.items():
    matches = df_all[df_all["text"].str.contains(pattern, case=False, na=False)]
    print(f"{target:<25}: {len(matches)} rows found")
    if len(matches) > 0:
        print("   Sample:", matches["text"].iloc[0][:100], "...")

=== SCRIPT BREAKDOWN FOR 4 DIRECT MATCHES ===
script_type             Mixed (Devanagari + Roman)  \
category                                             
Municipal Certificates                         124   
Roads & Infrastructure                         119   
Sanitation & Garbage                           143   
Water Supply                                   140   

script_type             Roman (English or Hinglish)  
category                                             
Municipal Certificates                           96  
Roads & Infrastructure                          101  
Sanitation & Garbage                             95  
Water Supply                                    120  

=== HIDDEN KEYWORD MATCHES ACROSS ENTIRE DATASET ===
Streetlights             : 48 rows found
   Sample: Sir/Madam, बिजली का खंभा टूट कर सड़क पर गिरा है Gandhi Nagar में, दुर्घटना हो सकती है, कृपया जल्दी ह ...
Drainage & Sewage        : 63 rows found
   Sample: Sewage line Kotwali area mein block ho gay

In [7]:
import pandas as pd

df_bbmp = pd.read_csv("../data/raw/bbmp_grievances.csv", encoding="utf-8", low_memory=False)

print("Shape (rows, columns):", df_bbmp.shape)
print("Columns:", df_bbmp.columns.tolist())

print("\nMissing values:")
print(df_bbmp.isnull().sum())

print("\nFirst 5 rows:")
display(df_bbmp.head(5))

Shape (rows, columns): (126974, 8)
Columns: ['Complaint ID', 'Category', 'Sub Category', 'Grievance Date', 'Ward Name', 'Grievance Status', 'Staff Remarks', 'Staff Name']

Missing values:
Complaint ID         0
Category             0
Sub Category         0
Grievance Date       0
Ward Name            0
Grievance Status    54
Staff Remarks       54
Staff Name          54
dtype: int64

First 5 rows:


,Complaint ID,Category,Sub Category,Grievance Date,Ward Name,Grievance Status,Staff Remarks,Staff Name
0,20771690,Electrical,Street Light Not Working,2025-06-19 10:39:00.000000000,Jagajeevanram Nagar,Registered,1st Assignment Based on Ward Mapping,syed zameer/JE
1,20771689,Electrical,Street Light Not Working,2025-06-19 10:36:00.000000000,Kammanahalli,Registered,1st Assignment Based on Ward Mapping,Vinay Kumar/AE
2,20771688,Solid Waste (Garbage) Related,Garbage dump,2025-06-19 10:35:00.000000000,Banaswadi,Registered,1st Assignment Based on Ward Mapping,Marshal Ward No 27/Marshal
3,20771687,Electrical,Street Light Not Working,2025-06-19 10:34:00.000000000,Gandhi Nagar,Registered,1st Assignment Based on Ward Mapping,syed zameer/JE
4,20771686,Solid Waste (Garbage) Related,Garbage dump,2025-06-19 10:34:00.000000000,Kumaraswamy Layout,Registered,1st Assignment Based on Ward Mapping,Marshal Ward No 181/Marshal
